# Router evaluation — does the system route questions to the right paradigm? (RQ1)

The capability-boundary claim (RQ1) rests on a **router**: the Layer-1 aggregate detector in
`rag_echr_ris.ipynb` decides whether a question is answerable by top-k retrieval (Bucket 1)
or must be **routed away from generation** (Buckets 2/3 — aggregate questions retrieval
would fabricate). Until now that router was 36 regex patterns with **no measurement**.
This notebook turns the design into a result:

- a **frozen gold question set** (60 questions, 20 per bucket, ~half German) → `data/`,
- routing **accuracy overall / per bucket / per language**,
- the two error rates that matter:
  - **fabrication risk** — Bucket-3 (content-aggregate) questions wrongly sent to retrieval;
    the system would then generate a corpus-wide claim from ~6 chunks,
  - **over-abstention** — Bucket-1 questions wrongly refused,
- a verbatim **failure table** (what to fix or to report as limitation),
- a report → `reports/router_evaluation_report.md`.

**Scope note.** The router is deliberately **binary** (retrieval vs aggregate). Separating
Bucket 2 (metadata aggregate → SQL over validated columns) from Bucket 3 (content aggregate →
extraction or abstention) happens downstream in the query layer, which knows which columns
exist. The router's only job — and the only thing measured here — is keeping aggregate
questions away from generation. The gold set still carries 2 vs 3 labels so the downstream
split stays measurable later.

## 1. Load the router from the RAG notebook (single source of truth)
The Layer-1 cell (`AGGREGATE_PATTERNS` + `aggregate_trigger`) is exec'd straight out of
`rag_echr_ris.ipynb`, so this evaluation can never drift from the deployed patterns.

In [ ]:
import json
import re
from pathlib import Path

DATA_DIR   = Path("../data")
REPORT_DIR = Path("../reports")
RAG_NB     = Path("rag_echr_ris.ipynb")
GOLD_OUT   = DATA_DIR / "router_gold_questions.csv"
REPORT_OUT = REPORT_DIR / "router_evaluation_report.md"

nb = json.loads(RAG_NB.read_text())
router_src = None
for c in nb["cells"]:
    src = "".join(c["source"])
    if c["cell_type"] == "code" and "AGGREGATE_PATTERNS = [" in src:
        router_src = src
        break
assert router_src is not None, "router cell not found in rag_echr_ris.ipynb"
exec(router_src)   # defines AGGREGATE_PATTERNS, _AGG_RE, aggregate_trigger, abstain messages
print(f"router loaded from {RAG_NB.name}: {len(AGGREGATE_PATTERNS)} patterns")

Type-1 detector ready (36 patterns)
router loaded from rag_echr_ris.ipynb: 36 patterns


## 2. Frozen gold question set — 60 questions, 20 per bucket, EN + DE
Buckets follow the RQ typology (`26.06.md`):

- **1 — explanatory / comparative / framing** → retrieval answers with citations,
- **2 — metadata aggregate** → computable by `groupby` over validated columns,
- **3 — content aggregate** → needs per-case extraction from prose; retrieval must NOT answer.

The set deliberately contains *hard* phrasings (aggregate intent without a trigger word —
e.g. *"Which importance level is most common…"*, *"Do married parents win more often…"*) so
the evaluation can fail honestly. Questions were written against the corpus topics
(contact, custody, alienation, Kindeswohl) in each court's register.

In [ ]:
# (question, gold_bucket 1|2|3, lang)
GOLD = [
    # ---- Bucket 1 — explanatory / comparative (EN) ----
    ("What positive obligations does the State have to enforce contact rights?", 1, "en"),
    ("How does the ECHR assess a child's refusal to see a parent?", 1, "en"),
    ("What role does the passage of time play in parent-child reunification cases?", 1, "en"),
    ("Under what conditions is coercive enforcement of contact against the child's will justified?", 1, "en"),
    ("How do courts weigh the best interests of the child against a parent's contact rights?", 1, "en"),
    ("What does the Court consider an effective remedy in contact-enforcement cases?", 1, "en"),
    ("When can custody be transferred to the other parent because of alienating behaviour?", 1, "en"),
    ("What is the margin of appreciation of domestic courts in custody disputes?", 1, "en"),
    ("How is expert psychological evidence treated in alienation cases?", 1, "en"),
    ("What weight is given to the child's expressed wishes in contact proceedings?", 1, "en"),
    # ---- Bucket 1 (DE) ----
    ("Wann ist von einer Vollzugsma\u00dfnahme abzusehen, wenn sie dem Kindeswohl widerspricht?", 1, "de"),
    ("Welche Voraussetzungen gelten f\u00fcr die gemeinsame Obsorge nach Trennung der Eltern?", 1, "de"),
    ("Wie begr\u00fcnden Gerichte die Einschr\u00e4nkung des Kontaktrechts?", 1, "de"),
    ("Welche Rolle spielt der Loyalit\u00e4tskonflikt des Kindes bei Kontaktrechtsentscheidungen?", 1, "de"),
    ("Unter welchen Voraussetzungen kann einem Elternteil die Obhut entzogen werden?", 1, "de"),
    ("Wie wird Bindungstoleranz in der Rechtsprechung bewertet?", 1, "de"),
    ("Was versteht die Rechtsprechung unter Kindeswohlgef\u00e4hrdung?", 1, "de"),
    ("Wann kann das Kontaktrecht gegen den Willen des Kindes durchgesetzt werden?", 1, "de"),
    ("Wie unterscheidet sich die Behandlung der Entfremdung vor dem OGH und dem EGMR?", 1, "de"),
    ("Welche Bedeutung hat die Beeinflussung des Kindes durch einen Elternteil f\u00fcr die Obsorgeentscheidung?", 1, "de"),
    # ---- Bucket 2 — metadata aggregates (EN) ----
    ("How many cases against Norway are in the corpus?", 2, "en"),
    ("Which respondent state has the most cases on contact rights?", 2, "en"),
    ("What proportion of merits judgments found a violation of Article 8?", 2, "en"),
    ("How many judgments were delivered after 2020?", 2, "en"),
    ("What is the number of communicated cases per country?", 2, "en"),
    ("How has the number of contact-rights cases changed over the years?", 2, "en"),
    ("What percentage of the corpus are admissibility decisions?", 2, "en"),
    ("Which importance level is most common among the judgments?", 2, "en"),
    ("How many Swiss decisions come from the canton of Zurich?", 2, "en"),
    ("What is the average time between communication of a case and the judgment?", 2, "en"),
    # ---- Bucket 2 (DE) ----
    ("Wie viele F\u00e4lle betreffen \u00d6sterreich?", 2, "de"),
    ("Wie hoch ist der Anteil der Verletzungsurteile?", 2, "de"),
    ("Wie oft wurde Artikel 8 verletzt?", 2, "de"),
    ("Wie viele Entscheidungen stammen aus dem Jahr 2023?", 2, "de"),
    ("Welcher Staat hat die meisten Verurteilungen?", 2, "de"),
    ("Wie hat sich die Zahl der Kontaktrechtsf\u00e4lle \u00fcber die Jahre entwickelt?", 2, "de"),
    ("Wie viele Urteile ergingen nach 2020?", 2, "de"),
    ("Wieviel Prozent der F\u00e4lle wurden f\u00fcr unzul\u00e4ssig erkl\u00e4rt?", 2, "de"),
    ("Aus welchem Kanton stammen die meisten Schweizer Entscheidungen?", 2, "de"),
    ("Wie ist die Gesamtzahl der anh\u00e4ngigen Verfahren?", 2, "de"),
    # ---- Bucket 3 — content aggregates (EN) ----
    ("How many cases involve an allegation of parental alienation?", 3, "en"),
    ("In what proportion of cases did the mother receive custody?", 3, "en"),
    ("How often do courts order supervised contact?", 3, "en"),
    ("How many children refused contact with their father across the corpus?", 3, "en"),
    ("What percentage of cases mention a psychological expert report?", 3, "en"),
    ("Do married parents win contact disputes more often than unmarried ones?", 3, "en"),
    ("How many cases involve grandparents seeking contact?", 3, "en"),
    ("What share of alienation allegations were upheld by the court?", 3, "en"),
    ("On average, how many years do contact-enforcement proceedings last?", 3, "en"),
    ("In most cases, which parent is found responsible for the alienation?", 3, "en"),
    # ---- Bucket 3 (DE) ----
    ("Wie viele Kinder leben nach der Entscheidung bei der Mutter?", 3, "de"),
    ("Wie h\u00e4ufig wird ein begleiteter Kontakt angeordnet?", 3, "de"),
    ("In welchem Anteil der F\u00e4lle wurde die Entfremdung vom Gericht festgestellt?", 3, "de"),
    ("Wie oft folgen die Gerichte dem Sachverst\u00e4ndigengutachten?", 3, "de"),
    ("Wie viele F\u00e4lle betreffen Gro\u00dfeltern als Kontaktwerber?", 3, "de"),
    ("Bekommen verheiratete Eltern in der Regel eher das Sorgerecht?", 3, "de"),
    ("Wie viele Verfahren dauerten l\u00e4nger als f\u00fcnf Jahre?", 3, "de"),
    ("Welcher Elternteil wird in den meisten F\u00e4llen f\u00fcr die Entfremdung verantwortlich gemacht?", 3, "de"),
    ("Wie hoch ist die durchschnittliche Verfahrensdauer in Kontaktrechtsf\u00e4llen?", 3, "de"),
    ("Wie oft wird das Kindeswohl als Hauptgrund f\u00fcr die Kontaktverweigerung genannt?", 3, "de"),
]

import pandas as pd
gold = pd.DataFrame(GOLD, columns=["question", "bucket", "lang"])
gold["gold_route"] = gold.bucket.map({1: "retrieval", 2: "aggregate", 3: "aggregate"})
gold.to_csv(GOLD_OUT, index=False)
print(f"gold set: {len(gold)} questions -> {GOLD_OUT.name}")
print(gold.groupby(["bucket", "lang"]).size().to_string())

gold set: 60 questions -> router_gold_questions.csv
bucket  lang
1       de      10
        en      10
2       de      10
        en      10
3       de      10
        en      10


## 3. Run the router over the gold set + score

In [ ]:
gold["trigger"] = gold.question.map(aggregate_trigger)
gold["routed"] = gold.trigger.map(lambda t: "aggregate" if t else "retrieval")
gold["correct"] = gold.routed == gold.gold_route

acc = gold.correct.mean()
print(f"ROUTING ACCURACY (aggregate vs retrieval): {acc:.3f}  (n={len(gold)})\n")

print("accuracy per bucket:")
print((gold.groupby("bucket").correct.agg(["mean", "size"]).rename(
    columns={"mean": "acc", "size": "n"})).round(3).to_string())
print("\naccuracy per language:")
print((gold.groupby("lang").correct.agg(["mean", "size"]).rename(
    columns={"mean": "acc", "size": "n"})).round(3).to_string())
print("\naccuracy per bucket x language:")
print((gold.groupby(["bucket", "lang"]).correct.mean().unstack().round(3)).to_string())

# the two error rates that matter
b3 = gold[gold.bucket == 3]
b1 = gold[gold.bucket == 1]
fabrication_risk = (b3.routed == "retrieval").mean()
over_abstention = (b1.routed == "aggregate").mean()
print(f"\nFABRICATION RISK  — Bucket-3 wrongly sent to retrieval: {fabrication_risk:.3f} "
      f"({int((b3.routed == 'retrieval').sum())}/{len(b3)})")
print(f"OVER-ABSTENTION   — Bucket-1 wrongly refused           : {over_abstention:.3f} "
      f"({int((b1.routed == 'aggregate').sum())}/{len(b1)})")

# --- why n=60: design grid + what it buys statistically -------------------------------
# 60 = 3 buckets x 2 languages x 10 questions: every bucket-language cell has equal weight,
# and 10 hand-authored questions per cell is the practical bound for deliberately *hard*,
# non-redundant items. The Wilson intervals below state what n=60 can and cannot claim;
# the gold set is a frozen CSV, so growing it later never invalidates old runs.
import math

def wilson(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, centre - half), min(1.0, centre + half))

acc_k = int(gold.correct.sum())
lo, hi = wilson(acc_k, len(gold))
print(f"\n95% Wilson CIs (what n=60 buys):")
print(f"  accuracy         {acc_k}/{len(gold)} = {acc_k/len(gold):.3f}  CI [{lo:.2f}, {hi:.2f}]")
fr_k = int((b3.routed == "retrieval").sum())
lo3, hi3 = wilson(fr_k, len(b3))
print(f"  fabrication risk {fr_k}/{len(b3)} = {fr_k/len(b3):.3f}  CI [{lo3:.2f}, {hi3:.2f}]  <- per-bucket n=20 is the binding constraint")
oa_k = int((b1.routed == "aggregate").sum())
lo1, hi1 = wilson(oa_k, len(b1))
print(f"  over-abstention  {oa_k}/{len(b1)} = {oa_k/len(b1):.3f}  CI [{lo1:.2f}, {hi1:.2f}]")

ROUTING ACCURACY (aggregate vs retrieval): 0.950  (n=60)

accuracy per bucket:
         acc   n
bucket          
1       1.00  20
2       0.95  20
3       0.90  20

accuracy per language:
        acc   n
lang           
de    0.967  30
en    0.933  30

accuracy per bucket x language:
lang     de   en
bucket          
1       1.0  1.0
2       1.0  0.9
3       0.9  0.9

FABRICATION RISK  — Bucket-3 wrongly sent to retrieval: 0.100 (2/20)
OVER-ABSTENTION   — Bucket-1 wrongly refused           : 0.000 (0/20)

95% Wilson CIs (what n=60 buys):
  accuracy         57/60 = 0.950  CI [0.86, 0.98]
  fabrication risk 2/20 = 0.100  CI [0.03, 0.30]  <- per-bucket n=20 is the binding constraint
  over-abstention  0/20 = 0.000  CI [0.00, 0.16]


## 4. Failure table — every misrouted question, verbatim
These are the thing to act on: fabrication-risk failures need a new pattern (or a semantic
fallback); over-abstention failures may be acceptable (the router is *biased to over-abstain*
by design).

In [ ]:
fails = gold[~gold.correct].copy()
fails["severity"] = fails.bucket.map({3: "FABRICATION RISK", 2: "missed aggregate", 1: "over-abstention"})
if fails.empty:
    print("no routing failures on the gold set")
else:
    for _, r in fails.sort_values("bucket", ascending=False).iterrows():
        print(f"[{r.severity:16s}] bucket={r.bucket} lang={r.lang} routed={r.routed}")
        print(f"    Q: {r.question}")
        print(f"    trigger: {r.trigger!r}")
print(f"\n{len(fails)} failures / {len(gold)} questions")

[FABRICATION RISK] bucket=3 lang=en routed=retrieval
    Q: Do married parents win contact disputes more often than unmarried ones?
    trigger: None
[FABRICATION RISK] bucket=3 lang=de routed=retrieval
    Q: Welcher Elternteil wird in den meisten Fällen für die Entfremdung verantwortlich gemacht?
    trigger: None
[missed aggregate] bucket=2 lang=en routed=retrieval
    Q: Which importance level is most common among the judgments?
    trigger: None

3 failures / 60 questions


## 5. Report → `reports/router_evaluation_report.md`

In [ ]:
lines = ["# Router evaluation report (RQ1 — capability boundary)\n\n"]
lines.append(f"- gold set: **{len(gold)}** questions (frozen: `{GOLD_OUT.name}`), "
             f"20 per bucket, {int((gold.lang == 'de').sum())} German / "
             f"{int((gold.lang == 'en').sum())} English\n"
             f"- router: the deployed Layer-1 detector exec'd from `rag_echr_ris.ipynb` "
             f"({len(AGGREGATE_PATTERNS)} patterns, binary retrieval-vs-aggregate)\n\n")
lines.append(f"## Headline numbers\n"
             f"- routing accuracy: **{acc:.3f}**\n"
             f"- fabrication risk (Bucket-3 sent to retrieval): **{fabrication_risk:.3f}** "
             f"({int((b3.routed == 'retrieval').sum())}/{len(b3)})\n"
             f"- over-abstention (Bucket-1 refused): **{over_abstention:.3f}** "
             f"({int((b1.routed == 'aggregate').sum())}/{len(b1)})\n\n")
lines.append("## Accuracy per bucket x language\n\n")
tbl = gold.groupby(["bucket", "lang"]).correct.mean().unstack().round(3)
cols = list(tbl.columns)
lines.append("| bucket | " + " | ".join(cols) + " |\n")
lines.append("|" + "---|" * (len(cols) + 1) + "\n")
for b, row in tbl.iterrows():
    lines.append(f"| {b} | " + " | ".join(str(row[c]) for c in cols) + " |\n")
lines.append("\n")
lines.append("## Failures (verbatim)\n\n")
if fails.empty:
    lines.append("none\n")
else:
    for _, r in fails.sort_values("bucket", ascending=False).iterrows():
        lines.append(f"- **{r.severity}** (bucket {r.bucket}, {r.lang}): \u201c{r.question}\u201d\n")
lines.append("\n## Why n=60 (design + statistics)\n"
             "- 60 = 3 buckets \u00d7 2 languages \u00d7 10 hand-authored questions: equal-weight "
             "cells; 10 non-redundant *hard* items per cell is the practical authoring bound.\n"
             "- What n=60 buys (95% Wilson): accuracy 0.95 \u2192 CI [0.86, 0.98]; fabrication "
             "risk 0.10 at per-bucket n=20 \u2192 CI [0.03, 0.30] \u2014 stated, not hidden: the "
             "headline is tight, per-bucket rates are indicative. The gold set is a frozen "
             "CSV; it can be grown without invalidating prior runs.\n")
lines.append("\n## Reading\n"
             "- The router is binary by design; Bucket 2 vs 3 is separated downstream by the "
             "query layer (which knows the validated columns). This report measures the "
             "boundary that matters for faithfulness: **no aggregate question may reach "
             "generation**.\n"
             "- Failures where aggregate intent carries no lexical trigger "
             "(e.g. \u201cmost common\u201d, \u201cmore often than\u201d) are the known limit of a "
             "pattern router \u2014 candidates for a semantic-routing extension, or reported as "
             "a stated limitation.\n")
REPORT_OUT.write_text("".join(lines), encoding="utf-8")
print("wrote", REPORT_OUT)

wrote ../reports/router_evaluation_report.md
